In [ ]:
def _fact_checks(df, running_date):
    """Return [(name, passed, detail)] for the gold fact, scoped to the run day.
 
    Gold accumulates history (one replaceWhere partition per report_date), so the table
    legitimately holds many dates. We therefore validate only the running_date slice:
    the run must have produced clean rows for its own day. Cross-day partitions from
    prior runs are out of scope."""
    from pyspark.sql import functions as F
 
    rd = str(running_date)[:10]
    day = df.where(F.col("report_date").cast("string") == F.lit(rd))
    total = day.count()
    a = day.agg(
        F.sum(F.when(F.col("transaction_id").isNull(), 1).otherwise(0)).alias("null_txn"),
        F.sum(F.when(F.col("product_id").isNull(), 1).otherwise(0)).alias("null_prod"),
        F.sum(F.when(F.col("report_date").isNull(), 1).otherwise(0)).alias("null_date"),
        F.sum(F.when((F.col("delivery_status") == "completed") & F.col("final_amount").isNull(), 1)
              .otherwise(0)).alias("null_amt"),
    ).first()
    # Grain guard: the fact's grain is one row per (transaction, product, line#).
    # gold's _merge left-joins delivery TWICE (by transaction_id and by order_no) and
    # promotions/points by id — any upstream duplicate fans out and silently multiplies
    # revenue. Row count vs distinct grain keys catches that class of bug loudly.
    grain_dups = total - day.select("transaction_id", "product_id",
                                    "row_num_transaction_id").distinct().count()
    return [
        (f"row_count > 0 for {rd}", total > 0, f"rows={total}"),
        ("no null transaction_id", (a["null_txn"] or 0) == 0, f"nulls={a['null_txn']}"),
        ("no null product_id", (a["null_prod"] or 0) == 0, f"nulls={a['null_prod']}"),
        ("no null report_date", (a["null_date"] or 0) == 0, f"nulls={a['null_date']}"),
        ("completed final_amount not null", (a["null_amt"] or 0) == 0, f"nulls={a['null_amt']}"),
        ("grain unique (transaction_id, product_id, row_num)", grain_dups == 0,
         f"duplicate_rows={grain_dups}"),
    ]

In [ ]:
def run_dq(ctx) -> None:
    """Read gold.fact_eod_sale_product for the run and raise if any check fails."""
    df = ctx.spark.read.table(ctx.gold("fact_eod_sale_product"))
    results = _fact_checks(df, ctx.running_date)
    for name, ok, detail in results:
        ctx.logger.info(f"[DQ] {'PASS' if ok else 'FAIL'} — {name} ({detail})")
    failed = [f"{n} ({d})" for n, ok, d in results if not ok]
    if failed:
        raise ValueError("DQ FAILED: " + " | ".join(failed))
    ctx.logger.info(f"[DQ] all {len(results)} checks passed for {str(ctx.running_date)[:10]}")